In [10]:
import pandas as pd
from datetime import datetime, timedelta
import pandas as pd 
df_products = pd.read_csv("data/process/catusita_consolidated.csv")
df_products['fecha'] = pd.to_datetime(df_products['fecha'], dayfirst=True)
df_predictions = pd.read_csv("data/cleaned/predictions.csv")
df_predictions['date'] = pd.to_datetime(df_predictions['date'])
df_predictions = df_predictions.rename(columns={'sku': 'articulo'})
df_products['year_month'] = (df_products['fecha'].dt.year.map(str) + '-' + df_products['fecha'].dt.month.map(lambda x: f"0{x}" if x < 10 else str(x)))

In [ ]:
ultima_fecha = df_products.groupby('articulo')['fecha'].max().reset_index()
df_ultima_fuente = df_products.merge(ultima_fecha, on=['articulo', 'fecha'], how='inner')
df_products = df_products.merge(
    df_ultima_fuente[['articulo', 'fuente_suministro']],
    on='articulo',
    how='left',
    suffixes=('', '_ultima')
)
df_products.rename(columns={'fuente_suministro_ultima': 'fuente_suministro_ultima'}, inplace=True)
df_products[df_products['articulo']=='100-3l'][['fecha', 'articulo', 'fuente_suministro', 'fuente_suministro_ultima']]


,fecha,articulo,fuente_suministro,fuente_suministro_ultima
262690,2022-01-15,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
269483,2022-06-16,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
275861,2022-11-11,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
330367,2022-11-04,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
366906,2022-09-15,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
413327,2022-03-17,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
432812,2022-06-27,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
433474,2022-09-22,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
452646,2022-05-16,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas
509276,2022-03-08,100-3l,birene - cigueñales-culatas-ejes de levas,ishikawa - cigueñales-culatas-ejes de levas


In [66]:
prev_day = datetime.now() - timedelta(days=1)
df_products['fecha'] = pd.to_datetime(df_products['fecha'], dayfirst=True)
df_products['mes_limite_6m'] = prev_day - pd.DateOffset(months=6)
df_filtrado_fecha = df_products[df_products['fecha'] >= df_products['mes_limite_6m']]
df_products = df_products[df_products['fecha'].dt.year == 2024]
demanda_sum = df_products.groupby('articulo')['venta_usd'].sum().reset_index()
numero_anos = len(df_products['fecha'].dt.year.unique())
demanda_sum["venta_usd"] = demanda_sum["venta_usd"]
demanda_sum
# demanda_sum.columns = ['articulo', 'demanda_promedio_anual_usd']
# df = df.merge(demanda_sum, on='articulo', how='left')
# demanda_sum


,articulo,venta_usd
0,00979brge,15.25
1,01017572b,27.12
2,01095jg2,20.34
3,01884brag,371.10
4,100-1kz-t,2728.81
...,...,...
5311,zx203,84.31
5312,zx275,7.80
5313,zxed1,2440.67
5314,zxelru1,238.98


In [7]:
prev_day = datetime.now() - timedelta(days=1)
df_products['mes_limite_6m'] = prev_day - pd.DateOffset(months=6)
df_filtro = df_products[df_products['fecha'] >= df_products['mes_limite_6m']]
df_merge = df_predictions.copy()
df = df_merge.merge(
    df_filtro[['articulo','fuente_suministro', 'venta_usd']].drop_duplicates(), 
    how='left', 
    on = 'articulo'
)
df_demanda_promedio = (
    df.groupby('articulo')['venta_usd']
    .sum()
    .reset_index()
)
# df_demanda_promedio['venta_usd_prom'] = df_demanda_promedio['venta_usd']/6
# df_products = df_products.merge(df_demanda_promedio, on='articulo', how='left')

In [8]:
lista_articulos_unicos = df_products[df_products['fuente_suministro']=='birene - cigueñales-culatas-ejes de levas']['articulo'].unique()
df_filtro[(df_filtro['articulo'].isin(lista_articulos_unicos))&(df_filtro['venta_usd']==211.86)]

,fecha,documento,articulo,codigo,cantidad,transacciones,venta_pen,venta_usd,fuente_suministro,costo,lt,year_month,fuente_suministro_ultima,mes_limite_6m
1092719,2025-01-29,f001-01-0060234,100-4d56t,NaN,2.0,1,792.99,211.86,ishikawa - cigueñales-culatas-ejes de levas,615.61,NaN,2025-01,ishikawa - cigueñales-culatas-ejes de levas,2024-09-12 14:26:02.656015
1093293,2025-01-29,f001-01-0060234,100-navara,NaN,1.0,1,792.99,211.86,ishikawa - cigueñales-culatas-ejes de levas,552.47,NaN,2025-01,ishikawa - cigueñales-culatas-ejes de levas,2024-09-12 14:26:02.656015
1093642,2025-01-29,f001-01-0060234,100-yd25,NaN,1.0,1,792.99,211.86,ishikawa - cigueñales-culatas-ejes de levas,611.67,NaN,2025-01,ishikawa - cigueñales-culatas-ejes de levas,2024-09-12 14:26:02.656015
